In [7]:
import json
import os

BJX_JSON = "bjx_raw.json"
EXTRA_JSON = "extra_raw.json"
RAW_JSON = "surnames_raw.json"
OUT_HTML = "output/index.html"
DB_PATH = None
DB_LABEL = "cbdb_20260808.sqlite3（2026-08-08 版）"

bjx = json.load(open(BJX_JSON, encoding="utf-8"))
extra = json.load(open(EXTRA_JSON, encoding="utf-8"))
surnames = json.load(open(RAW_JSON, encoding="utf-8"))

print("BJX 行数  :", len(bjx), "| 有数据:", sum(1 for x in bjx if x[1] > 0))
print("EXTRA 行数:", len(extra), "| 人数:", sum(x[1] for x in extra))
print("原始键数  :", len(surnames), "| 有姓总人数:", sum(surnames.values()))

BJX 行数  : 504 | 有数据: 475
EXTRA 行数: 2358 | 人数: 29087
原始键数  : 2874 | 有姓总人数: 647915


In [8]:
def fmt(n):
    return f"{n:,}"

# RAW：Top 50（有数据的序位按人数降序）
raw_rows = sorted([x for x in bjx if x[1] > 0], key=lambda x: (-x[1], x[0]))[:50]
RAW = [[s, n, b] for s, n, b in raw_rows]

BJX = bjx
EXTRA = [[s, n, "—"] for s, n in extra]

TOTAL_NAMED = sum(x[1] for x in BJX) + sum(x[1] for x in extra)
TOP20_SUM = sum(x[1] for x in RAW[:20])

TOTAL_PERSONS = None
if DB_PATH and os.path.exists(DB_PATH):
    import sqlite3
    con = sqlite3.connect("file:" + DB_PATH + "?mode=ro", uri=True)
    TOTAL_PERSONS = con.execute("SELECT COUNT(*) FROM BIOG_MAIN").fetchone()[0]
    con.close()
if not TOTAL_PERSONS:
    TOTAL_PERSONS = TOTAL_NAMED
    print("提示：未提供 DB_PATH，人物总数暂用有姓人数代替（= 有姓总人数）")

print("RAW 行数  :", len(RAW), "| Top1:", RAW[0])
print("BJX 合计  :", fmt(sum(x[1] for x in BJX)))
print("EXTRA 合计:", fmt(sum(x[1] for x in extra)))
print("TOTAL_NAMED:", fmt(TOTAL_NAMED), "| TOTAL_PERSONS:", fmt(TOTAL_PERSONS))
print("TOP20_SUM :", fmt(TOP20_SUM))

提示：未提供 DB_PATH，人物总数暂用有姓人数代替（= 有姓总人数）
RAW 行数  : 50 | Top1: ['李', 41344, 4]
BJX 合计  : 618,828
EXTRA 合计: 29,087
TOTAL_NAMED: 647,915 | TOTAL_PERSONS: 647,915
TOP20_SUM : 318,837


In [6]:
template = open("template.html", encoding="utf-8").read()
print("模板已加载: template.html | 长度:", len(template), "字节")

js_arr = lambda rows: json.dumps(rows, ensure_ascii=False)

repl = {
    "__RAW__": js_arr(RAW),
    "__BJX__": js_arr(BJX),
    "__EXTRA__": js_arr(EXTRA),
    "__TOTAL_PERSONS__": str(TOTAL_PERSONS),
    "__TOTAL_NAMED__": str(TOTAL_NAMED),
    "__K1__": fmt(TOTAL_PERSONS),
    "__K2__": fmt(TOTAL_NAMED),
    "__K3__": fmt(TOP20_SUM),
    "__K3PCT__": f"{TOP20_SUM / TOTAL_NAMED * 100:.1f}",
    "__TOP1_NAME__": RAW[0][0],
    "__TOP1_N__": fmt(RAW[0][1]),
    "__TOP1_PCT__": f"{RAW[0][1] / TOTAL_NAMED * 100:.1f}",
    "__DB_LABEL__": DB_LABEL,
    "__TOTAL_PERSONS_FMT__": fmt(TOTAL_PERSONS),
    "__TOTAL_NAMED_FMT__": fmt(TOTAL_NAMED),
    "__EXTRA_COUNT__": str(len(extra)),
}

missing = [k for k in repl if k not in template]
assert not missing, "模板缺少占位符: " + ", ".join(missing)

html_out = template
for k, v in repl.items():
    html_out = html_out.replace(k, v)

# 替换后不得再残留占位符
leftover = [k for k in repl if k in html_out]
assert not leftover, "残留占位符: " + ", ".join(leftover)

os.makedirs(os.path.dirname(OUT_HTML) or ".", exist_ok=True)
with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(html_out)
print("已生成:", OUT_HTML, "| 大小:", len(html_out), "字节")

NameError: name 'template' is not defined